# Reproducing TG-CDDPM

**Hesam Afshar**  
**Course:** Bioinformatics, University of Tehran

This notebook reproduces the main workflow of **TG-CDDPM: text-guided antimicrobial peptides generation based on conditional denoising diffusion probabilistic model**. I first set up the files, data, and libraries needed for the original implementation. Then I ran the three stages of the method and trained them as far as the available computational resources allowed. Finally, generated peptide samples were evaluated using AM-Score.

The full training setup of the paper was too expensive for this course project. For this reason, some later steps use checkpoints released by the original authors. The notebook keeps these limitations clear so the results are not presented as a full end-to-end reproduction of the paper.


## Setup


The original TG-CDDPM repository is cloned first. The dataset and checkpoints used in the paper were stored in Google Drive for this Colab run, so Drive is mounted in the next step.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Clone repo
!git clone https://github.com/JunhangCao/TG-CDDPM.git

Cloning into 'TG-CDDPM'...
remote: Enumerating objects: 151, done.
remote: Counting objects: 100% (151/151), done.
remote: Compressing objects: 100% (147/147), done.
remote: Total 151 (delta 61), reused 54 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (151/151), 188.84 KiB | 23.61 MiB/s, done.
Resolving deltas: 100% (61/61), done.


### Install requirements

The requirements from the original repository and the additional libraries needed for this notebook are installed in this section.


In [ ]:
# install requirements
%%bash
iconv -f utf-16le -t utf-8 /content/TG-CDDPM/requirements.txt \
 | grep -a -vE 'file://|\\\\' \
 | sed 's/^mkl-fft==1\.3\.1/mkl-fft==1.3.14/' \
 > req-clean.txt
pip install -r req-clean.txt
rm req-clean.txt


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.9/31.9 MB 65.1 MB/s eta 0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.4/13.4 MB 9.5 MB/s eta 0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.7/145.7 kB 17.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 93.9 MB/s eta 0:00:00
  Preparing metad

ERROR: Could not find a version that satisfies the requirement mkl-service==2.4.0 (from versions: 2.4.2, 2.5.0, 2.5.2)
ERROR: No matching distribution found for mkl-service==2.4.0


In [ ]:
# install other useful libraries
!pip install info-nce-pytorch==0.1.4
!pip install mkl-service==2.5.2
!pip install datasets
!pip install jsonlines

  Using cached info_nce_pytorch-0.1.4-py3-none-any.whl.metadata (3.1 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.5/82.5 kB 10.6 MB/s eta 0:00:00


### Unzip data and checkpoints

The dataset and checkpoint files are extracted and placed in the paths used by the original implementation.


In [ ]:
# unzip dataset
%%bash
ZIP="/content/drive/MyDrive/dataset.zip"
DEST="/content/TG-CDDPM/dataset"

mkdir -p "$DEST"
unzip -q "$ZIP" -d "$DEST"

echo "Unzipped files:"
ls -1 "$DEST" | head


Unzipped files:
backup
public_database


In [ ]:
# unzip checkpoints
%%bash
ZIP="/content/drive/MyDrive/checkpoints.zip"
DEST="/content/TG-CDDPM/checkpoints"

mkdir -p "$DEST"
unzip -q "$ZIP" -d "$DEST"

echo "Unzipped files:"
ls -1 "$DEST" | head


Unzipped files:
bert
pep_encoder.pt
text_encoder.pt
translator.pt
wo_pretraining_uncondition.pt
wo_pretraining_w_inference.pt
wo_pretraining_wo_inference.pt
w_pretraining_uncondition.pt
w_pretraining_w_inference.pt
w_pretraining_wo_inference.pt


## Dataset

The data used in TG-CDDPM has two main parts.

For pre-training, about 5.5 million peptide sequences from UniProt are split using a rolling window of length 50 and step 1, producing 8,818,283 samples. Non-standard amino-acid symbols are removed, highly similar sequences are filtered with CD-HIT at 90% similarity, and shorter sequences are padded to length 50. This stage is used to learn general peptide sequence patterns.

For fine-tuning, the paper collects a balanced set of active antimicrobial peptides from DBAASP, dbAMP, CAMP, APD3, and DRAMP, together with inactive samples from AMPlify. Sequences longer than 50 are removed. The released data contains 116,924 text-sequence entries, and because several descriptions can be available for one peptide, the final text-sequence pairing contains 252,294 pairs. The paper uses 90% for training and 10% for evaluation.


In [ ]:
import json, collections

def read_jsonl(path):
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            j = json.loads(line)
            yield j['src'], j['trg']  # text, tokenized peptide (list or str)

# paths used by the code
train_path = '/content/TG-CDDPM/dataset/backup/ensemble_train.jsonl'
valid_path = '/content/TG-CDDPM/dataset/backup/ensemble_test.jsonl'

train = list(read_jsonl(train_path))
valid = list(read_jsonl(valid_path))
all_pairs = train + valid

# count unique sequences (collapse tokens to a single string)
def seq_key(trg):
    if isinstance(trg, list):
        return ' '.join(trg)
    return str(trg)

unique_seq = {}
for src, trg in all_pairs:
    unique_seq.setdefault(seq_key(trg), []).append(src)

print('Total records:', len(all_pairs))
print('Unique sequences:', len(unique_seq))
print('Avg prompts per sequence:', len(all_pairs) / len(unique_seq))


Total records: 252294
Unique sequences: 175684
Avg prompts per sequence: 1.4360670294392204


## Stage 1

Stage 1 aligns the target text and peptide sequence in a shared 256-dimensional latent space. Correct text-peptide pairs should be close to each other, while mismatched pairs should be farther apart.

The text side uses a SciBERT-based encoder with a 12-layer Transformer backbone, hidden size 768, and 12 attention heads. Its token representations are projected to 256 dimensions. The peptide side uses a lighter ProGen-based encoder with `n_embd=256`, `n_layer=2`, and sequence length 50. The two encoders are trained with an InfoNCE contrastive objective.

For this project, Stage 1 was trained for 50 epochs on the full dataset, which took about 17 hours.


In [ ]:
import os, sys
REPO_DIR = "/content/TG-CDDPM"
sys.path.append(REPO_DIR)
os.chdir(REPO_DIR)

import argparse
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import warnings
from torch.utils.data import DataLoader, Dataset

from transformers import AutoTokenizer, AutoModel, AutoConfig
from config.backen_config import ProGenConfig
from model.backend import ProGenForCausalLM
from config.base_config import add_dict_to_argparser
from info_nce import InfoNCE
from einops import reduce, rearrange

from utils.tokenizer import TextTokenizer

warnings.filterwarnings('ignore')


def create_argparser():
    defaults = dict(
        batch_size=32,
        clip_epoches=50,
        fac_epoches=10,
        lr=1e-4,
        weight_decay=0.999,
        valid_rate=0.2,
        vocab_size=21,
        n_positions=256,
        n_ctx=256,
        n_embd=256,
        n_layer=2,
        seq_len=[50],
        class_cond=True,
        vocab_path="./mapping/vocab.txt",
        save_path="./checkpoints/"
    )
    parser = argparse.ArgumentParser()
    add_dict_to_argparser(parser, defaults)
    return parser


def get_text_peptides(path):
    text_peptides = pd.read_csv(path)
    des = list(text_peptides['Description'])
    sequences = list(text_peptides['Sequence'])
    texts = []
    seqs = []
    for i in range(len(des)):
        text = str(des[i])
        texts.append(text)
        s = ""
        for j in range(len(sequences[i]) - 1):
            s += sequences[i][j] + ' '
        s += sequences[i][-1]
        seqs.append(s)
    return texts, seqs


def l2norm(t):
    return F.normalize(t, dim=-1)


def matrix_diag(t):
    device = t.device
    i, j = t.shape[-2:]
    num_diag_el = min(i, j)
    i_range = torch.arange(i, device=device)
    j_range = torch.arange(j, device=device)
    diag_mask = rearrange(i_range, 'i -> i 1') == rearrange(j_range, 'j -> 1 j')
    diag_el = t.masked_select(diag_mask)
    return rearrange(diag_el, '(b d) -> b d', d=num_diag_el)

def masked_mean(t, mask, dim=1, eps=1e-6):
    t = t.masked_fill(~mask, 0.)
    numer = t.sum(dim=dim)
    denom = mask.sum(dim=dim).clamp(min=eps)
    return numer / denom

def max_neg_value(dtype):
    return -torch.finfo(dtype).max

def log(t, eps = 1e-20):
    return torch.log(t + eps)


class TextEncoder(nn.Module):
    def __init__(self,
                 hidden_state_dim=256,
                 output_dim=256,
                 max_len=50):
        super().__init__()
        self.bert_config = AutoConfig.from_pretrained('allenai/scibert_scivocab_uncased',
                                                      cache_dir='./checkpoints/bert/scibert_scivocab_uncased')
        self.sci_bert = AutoModel.from_pretrained('allenai/scibert_scivocab_uncased',
                                                  cache_dir='./checkpoints/bert/scibert_scivocab_uncased')
        # self.ln = nn.Sequential(
        #     nn.LayerNorm(self.bert_config.hidden_size),
        #     nn.Linear(self.bert_config.hidden_size, hidden_state_dim),)
        self.ln = nn.Linear(self.bert_config.hidden_size, output_dim)
        self.mlp = nn.Sequential(
            nn.Linear(hidden_state_dim * max_len, hidden_state_dim),
            nn.GELU(),
            nn.LayerNorm(hidden_state_dim),
            nn.Linear(hidden_state_dim, output_dim),
        )

    def get_features(self, x):
        return self.ln(self.sci_bert(x)['last_hidden_state'])

    def forward(self, x):
        b = x.shape[0]
        hidden_states = self.sci_bert(x)
        hidden_states = hidden_states['last_hidden_state']
        hidden_states = self.ln(hidden_states)
        hidden_states = hidden_states.reshape(b, -1)
        hidden_states = self.mlp(hidden_states)
        return hidden_states


class PepEncoder(nn.Module):
    def __init__(self, config, max_len=50):
        super().__init__()
        # self.pep_encoder = AutoModel.from_pretrained("Rostlab/prot_bert",
        #                                              cache_dir='../checkpoints/bert/prot_bert')
        self.pep_encoder = ProGenForCausalLM(config)
        self.ln = nn.Sequential(
            nn.Linear(config.n_embd*max_len, config.n_embd),
            nn.GELU(),
            nn.LayerNorm(config.n_embd),
            nn.Linear(config.n_embd, config.n_embd),
        )

    def get_features(self, x, input_embeds=None):
        if input_embeds is not None:
            return self.pep_encoder(inputs_embeds=input_embeds)
        else:
            return self.pep_encoder(x)

    def forward(self, x, input_embeds=None):
        if input_embeds is not None:
            b = input_embeds.shape[0]
            hidden_states = self.pep_encoder(inputs_embeds=input_embeds)
        else:
            b = x.shape[0]
            hidden_states = self.pep_encoder(x)
        hidden_states = hidden_states.reshape(b, -1)
        hidden_states = self.ln(hidden_states)
        return hidden_states


class Facilitator(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.mlp = ProGenForCausalLM(config)

    def forward(self, x):
        output = self.mlp(inputs_embeds=x)
        return output


class BatchDataset(Dataset):
    def __init__(self, sentences):
        super().__init__()
        self.raw_dataset = sentences
        self.len = len(sentences['src'])

    def __len__(self):
        return self.len

    def __getitem__(self, idx):
        text = self.raw_dataset['src'][idx]
        # print(self.raw_dataset['train'][idx]['src'])
        peptide = self.raw_dataset['trg'][idx]
        return text, peptide


# data loading
train_texts, train_peptides = get_text_peptides('./dataset/backup/ensemble_train.csv')
test_texts, test_peptides = get_text_peptides('./dataset/backup/ensemble_test.csv')
args = create_argparser().parse_args(args=[])
device = ('cuda' if torch.cuda.is_available() else 'cpu')
text_tokenizer = TextTokenizer()
text_encoder = TextEncoder().to(device)
# text_encoder.load_state_dict(torch.load(args.save_path + 'text_encoder.pt'))
pep_tokenizer = AutoTokenizer.from_pretrained('./checkpoints/bert/prot_bert')
pep_config = ProGenConfig(vocab_size=pep_tokenizer.vocab_size,
                          n_positions=args.n_positions,
                          n_ctx=args.n_ctx,
                          n_embd=args.n_embd,
                          n_layer=args.n_layer)
# pep_tokenizer = PepTokenizer(args.vocab_path)

pep_encoder = PepEncoder(pep_config).to(device)
# pep_encoder.load_state_dict(torch.load(args.save_path + 'pep_encoder.pt'))
# train
text_tokens = text_tokenizer.batch_encode(train_texts)
pep_tokens = pep_tokenizer(train_peptides,
                            padding=True,
                            truncation=True,
                            return_tensors="pt",
                            max_length=50)['input_ids']
sentence = {'src': [], 'trg': []}
for i in range(len(text_tokens)):
    sentence['src'].append(text_tokens[i])
    sentence['trg'].append(pep_tokens[i])
dataset = BatchDataset(sentence)

train_dataloader = DataLoader(dataset,
                              shuffle=True,
                              num_workers=0,
                              batch_size=args.batch_size,
                              drop_last=True, )
# valid
test_text_tokens = text_tokenizer.batch_encode(test_texts)
test_pep_tokens = pep_tokenizer(test_peptides,
                            padding=True,
                            truncation=True,
                            return_tensors="pt",
                            max_length=50)['input_ids']
valid = {'src': [], 'trg': []}
for i in range(len(test_pep_tokens)):
    valid['src'].append(test_text_tokens[i])
    valid['trg'].append(test_pep_tokens[i])
valid_dataset = BatchDataset(valid)
valid_dataloader = DataLoader(valid_dataset,
                              shuffle=True,
                              num_workers=0,
                              batch_size=args.batch_size,
                              drop_last=True, )
infonce = InfoNCE()
optimizer1 = torch.optim.Adam(params=text_encoder.parameters(), lr=args.lr, weight_decay=args.weight_decay)
optimizer2 = torch.optim.Adam(params=pep_encoder.parameters(), lr=args.lr, weight_decay=args.weight_decay)
# text_encoder.load_state_dict(torch.load('../checkpoints/text_encoder.pt'))
# pep_encoder.load_state_dict(torch.load('../checkpoints/pep_encoder.pt'))

print('-------------------training-----------------------')
for e in range(args.clip_epoches):
    e_loss = 0
    for text, pep in train_dataloader:
        # optimizer3.zero_grad()
        optimizer1.zero_grad()
        optimizer2.zero_grad()
        text = text.to(device)
        pep = pep.to(device)
        text_hidden_states = text_encoder(text)
        # pep_input_embeds = pep_encoder.pep_encoder.get_input_embeddings()(pep)
        pep_hidden_states = pep_encoder(pep)
        loss = infonce(text_hidden_states, pep_hidden_states)
        e_loss += loss
        loss.backward()
        # optimizer3.step()
        optimizer1.step()
        optimizer2.step()
        torch.cuda.empty_cache()
    print("epoch :{}, clip training loss: {}".format(e + 1, e_loss / len(train_dataloader)))

    valid_loss = 0
    for valid_text, valid_pep in valid_dataloader:
        with torch.no_grad():
            valid_text = valid_text.to(device)
            valid_pep = valid_pep.to(device)
            valid_text_hidden_states = text_encoder(valid_text)
            valid_pep_hidden_states = pep_encoder(valid_pep)
            valid_text_hidden_states = valid_text_hidden_states.reshape(args.batch_size, -1)
            valid_pep_hidden_states = valid_pep_hidden_states.reshape(args.batch_size, -1)
            # valid_text_z = facilitator(valid_text_z)
            # valid_pep_z = facilitator(valid_pep_z)

            loss = infonce(valid_text_hidden_states, valid_pep_hidden_states)
            # loss2 = loss_fn(valid_pep_z, valid_text_z)
            # loss = (loss1 + loss2) / 2
            valid_loss += loss
    print("epoch :{}, clip valid loss: {}".format(e + 1, valid_loss / len(valid_dataloader)))
# save model parameters
torch.save(text_encoder.state_dict(), args.save_path + 'text_encoder.pt')
torch.save(pep_encoder.state_dict(), args.save_path + 'pep_encoder.pt')

print('--------------------done--------------------------')

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/442M [00:00<?, ?B/s]

ProGenForCausalLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.


-------------------training-----------------------
epoch :1, clip training loss: 2.9967942237854004
epoch :1, clip valid loss: 2.922065496444702
epoch :2, clip training loss: 2.9268763065338135
epoch :2, clip valid loss: 2.891082286834717
epoch :3, clip training loss: 2.884641408920288
epoch :3, clip valid loss: 2.8467700481414795
epoch :4, clip training loss: 2.871356964111328
epoch :4, clip valid loss: 2.8507912158966064
epoch :5, clip training loss: 2.8651115894317627
epoch :5, clip valid loss: 2.846799850463867
epoch :6, clip training loss: 2.8591582775115967
epoch :6, clip valid loss: 2.8735198974609375
epoch :7, clip training loss: 2.8570032119750977
epoch :7, clip valid loss: 2.8518826961517334
epoch :8, clip training loss: 2.8553647994995117
epoch :8, clip valid loss: 2.8377315998077393
epoch :9, clip training loss: 2.8535842895507812
epoch :9, clip valid loss: 2.844099760055542
epoch :10, clip training loss: 2.8529627323150635
epoch :10, clip valid loss: 2.8521568775177
epoch 

In [ ]:
import os
save_path = '/content/drive/MyDrive/TG-CDDPM/stage1-checkpoints/'
os.makedirs(save_path, exist_ok=True)
torch.save(text_encoder.state_dict(), save_path + 'text_encoder.pt')
torch.save(pep_encoder.state_dict(), save_path + 'pep_encoder.pt')

## Stage 2

Stage 2 learns a mapping from the text embedding to a peptide embedding. The TextEncoder and PepEncoder from Stage 1 are frozen, and a Translator/Adapter is trained on text-peptide pairs. The Translator is a small MLP with Linear, GELU, Linear, and Dropout layers. A small amount of noise is added to the text embedding during training, and the model learns to return an embedding close to the real peptide embedding.

The main dimensions follow Stage 1: `n_embd=256`, `n_layer=2`, `n_positions=256`, and sequence length 50.

The training loop for this stage was not available in the original repository, so I wrote it in this notebook. Because of limited time and computational resources, I trained this stage for 10 epochs, which took about 3 hours. To make the later results more meaningful, the released TextEncoder and PepEncoder checkpoints from the original authors were used here. My own Stage 1 checkpoints were also saved.


In [ ]:
import os, sys, time, random, json, math, numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
from config.backen_config import ProGenConfig
from train.TexPepAlignment import TextEncoder, PepEncoder, get_text_peptides
from model.backend import FacModel
from utils.tokenizer import TextTokenizer
from model.gaussian_diffusion import get_named_beta_schedule

REPO_DIR = "/content/TG-CDDPM"
sys.path.append(REPO_DIR)
os.chdir(REPO_DIR)

TRAIN_CSV = "./dataset/backup/ensemble_train.csv"
VALID_CSV = "./dataset/backup/ensemble_test.csv"

PRETRAINED_TEXT_ENCODER = "./checkpoints/text_encoder.pt"
PRETRAINED_PEP_ENCODER  = "./checkpoints/pep_encoder.pt"

SAVE_DIR = "./checkpoints"
SAVE_TRANSLATOR  = os.path.join(SAVE_DIR, "diffusion_model_facilitator.pt")
SAVE_LAST = os.path.join(SAVE_DIR, "diffusion_model_facilitator_last.pt")
RESUME_TRANSLATOR= None

PEP_TOKENIZER_PATH = "./checkpoints/bert/prot_bert"

# Model dims (must match Stage-1/3)
N_POSITIONS = 256
N_CTX = 256
N_EMBD = 256
N_LAYER = 2
MAX_TEXT_LEN= 50
MAX_PEP_LEN = 50

# Training
SEED = 1234
EPOCHS = 10
BATCH_SIZE = 128
LR = 5e-4
WEIGHT_DECAY = 0.0
GRAD_CLIP    = 1.0

# Diffusion
T_STEPS        = 500
NOISE_SCHEDULE = "linear"

USE_AMP = torch.cuda.is_available()
USE_COMPILE = False
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.benchmark = True

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed(SEED)

def cosine_sim(a, b):
    a = F.normalize(a, dim=-1); b = F.normalize(b, dim=-1)
    return (a * b).sum(dim=-1)

def build_alpha_bar(T, schedule="linear", device=DEVICE):
    betas = get_named_beta_schedule(schedule, T)
    betas = torch.tensor(betas, dtype=torch.float32, device=device)
    alphas = 1.0 - betas
    return torch.cumprod(alphas, dim=0)

def q_sample(z0, t, alpha_bar, noise=None):
    if noise is None:
        noise = torch.randn_like(z0)
    a_bar_t = alpha_bar[t].unsqueeze(1)
    return torch.sqrt(a_bar_t) * z0 + torch.sqrt(1.0 - a_bar_t) * noise

def safe_load(model, ckpt_path, name="model", drop_contains=("position_ids",)):
    if not os.path.isfile(ckpt_path):
        print(f"[{name}] WARNING: checkpoint not found at {ckpt_path}")
        return [], []
    sd = torch.load(ckpt_path, map_location="cpu")
    if isinstance(sd, dict) and "state_dict" in sd:
        sd = sd["state_dict"]
    if not isinstance(sd, dict):
        raise RuntimeError(f"[{name}] Invalid checkpoint format: {type(sd)}")
    sd = { (k[7:] if k.startswith("module.") else k): v for k, v in sd.items() }
    drop_keys = [k for k in list(sd.keys()) if any(s in k for s in drop_contains)]
    for k in drop_keys:
        sd.pop(k)
    missing, unexpected = model.load_state_dict(sd, strict=False)
    print(f"[{name}] loaded from {ckpt_path} (strict=False)")
    return missing, unexpected

#  Dataset
class PairDataset(Dataset):
    def __init__(self, csv_path, max_text_len=MAX_TEXT_LEN, max_pep_len=MAX_PEP_LEN, pep_tokenizer_name=PEP_TOKENIZER_PATH):
        texts, seqs_spaced = get_text_peptides(csv_path)
        self.text_tok = TextTokenizer(max_len=max_text_len)
        tok_name = pep_tokenizer_name
        if not os.path.isdir(tok_name):
            tok_name = "Rostlab/prot_bert"
        self.pep_tok = AutoTokenizer.from_pretrained(tok_name)
        self.text_ids = self.text_tok.batch_encode(texts)
        pep = self.pep_tok(seqs_spaced, padding=True, truncation=True, return_tensors="pt", max_length=max_pep_len)
        self.pep_ids = pep["input_ids"].long()

    def __len__(self):  return self.text_ids.size(0)
    def __getitem__(self, i):
        return self.text_ids[i].long(), self.pep_ids[i].long()

train_ds = PairDataset(TRAIN_CSV)
valid_ds = PairDataset(VALID_CSV)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True, drop_last=True)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True, drop_last=False)
print(f"Train pairs: {len(train_ds)} | Valid pairs: {len(valid_ds)}")

# Encoders
pep_vocab_size = AutoTokenizer.from_pretrained(
    PEP_TOKENIZER_PATH if os.path.isdir(PEP_TOKENIZER_PATH) else "Rostlab/prot_bert"
).vocab_size

pep_cfg = ProGenConfig(vocab_size=pep_vocab_size, n_positions=N_POSITIONS, n_ctx=N_CTX, n_embd=N_EMBD, n_layer=N_LAYER)

text_encoder = TextEncoder(hidden_state_dim=N_EMBD, output_dim=N_EMBD, max_len=MAX_TEXT_LEN).to(DEVICE).eval()
pep_encoder  = PepEncoder(pep_cfg, max_len=MAX_PEP_LEN).to(DEVICE).eval()

safe_load(text_encoder, PRETRAINED_TEXT_ENCODER, name="text_encoder", drop_contains=("position_ids",))
safe_load(pep_encoder,  PRETRAINED_PEP_ENCODER,  name="pep_encoder",  drop_contains=("position_ids","lm_head.weight","lm_head.bias"))

for p in text_encoder.parameters(): p.requires_grad = False
for p in pep_encoder.parameters():  p.requires_grad = False

with torch.no_grad():
    _ti = torch.zeros(2, MAX_TEXT_LEN, dtype=torch.long, device=DEVICE)
    _pi = torch.zeros(2, MAX_PEP_LEN,  dtype=torch.long, device=DEVICE)
    zt  = text_encoder(_ti);  zp = pep_encoder(_pi)
    assert zt.shape == zp.shape == (2, N_EMBD), f"Embed dim mismatch: text {zt.shape}, pep {zp.shape}, N_EMBD {N_EMBD}"
print(f"Encoders ready. Embedding dim = {N_EMBD}")

#  Translator and optimizer
translator = FacModel(pep_cfg).to(DEVICE)
if USE_COMPILE and hasattr(torch, "compile"):
    translator = torch.compile(translator)

opt    = torch.optim.AdamW(translator.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

alpha_bar = build_alpha_bar(T_STEPS, NOISE_SCHEDULE, device=DEVICE)
os.makedirs(SAVE_DIR, exist_ok=True)


Train pairs: 227064 | Valid pairs: 25230


ProGenForCausalLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.


[text_encoder] loaded from ./checkpoints/text_encoder.pt (strict=False)
[pep_encoder] loaded from ./checkpoints/pep_encoder.pt (strict=False)
Encoders ready. Embedding dim = 256


In [ ]:
# Training loop
start_epoch = 1
for epoch in range(start_epoch, EPOCHS + 1):
    translator.train()
    epoch_start = time.time()
    running_loss = 0.0
    num_batches = 0

    for (text_ids, pep_ids) in train_loader:
        text_ids = text_ids.to(DEVICE, non_blocking=True)
        pep_ids  = pep_ids.to(DEVICE,  non_blocking=True)

        with torch.no_grad():
            z_text = text_encoder(text_ids)   # (B, D)
            z_pep  = pep_encoder(pep_ids)     # (B, D)

        B = z_text.size(0)
        t = torch.randint(0, T_STEPS, (B,), device=DEVICE, dtype=torch.long)
        z_t = q_sample(z_text, t, alpha_bar)  # (B, D)

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            pred = translator(inputs_embeds=z_t.unsqueeze(1), timesteps=t).squeeze(1)
            loss = F.mse_loss(F.normalize(pred, dim=-1), F.normalize(z_pep, dim=-1))

        scaler.scale(loss).backward()
        if GRAD_CLIP and GRAD_CLIP > 0:
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(translator.parameters(), GRAD_CLIP)
        scaler.step(opt); scaler.update()
        opt.zero_grad(set_to_none=True)

        running_loss += float(loss.item())
        num_batches  += 1

    avg_train_loss = running_loss / max(1, num_batches)
    epoch_time = time.time() - epoch_start

    # Validation
    translator.eval()
    cs_vals = []
    with torch.no_grad():
        for text_ids, pep_ids in valid_loader:
            text_ids = text_ids.to(DEVICE); pep_ids = pep_ids.to(DEVICE)
            z_text = text_encoder(text_ids); z_pep = pep_encoder(pep_ids)
            t0 = torch.zeros(z_text.size(0), dtype=torch.long, device=DEVICE)
            out = translator(inputs_embeds=z_text.unsqueeze(1), timesteps=t0).squeeze(1)
            cs_vals.append(cosine_sim(out, z_pep).mean().item())
    val_cs = float(np.mean(cs_vals)) if cs_vals else -1e9

    print(f"[epoch {epoch}] train_loss={avg_train_loss:.6f}  val_cos={val_cs:.4f}  time={epoch_time:.1f}s")

    # Save last + best
    torch.save({"model": translator.state_dict(), "opt": opt.state_dict(),
                "epoch": epoch, "best_val": best_val}, SAVE_LAST)
    if val_cs > best_val:
        best_val = val_cs
        torch.save(translator.state_dict(), SAVE_TRANSLATOR)
        print(f"  ↳ new best saved: {SAVE_TRANSLATOR}  (best val_cos={best_val:.4f})")

print("Stage-2 training finished.")
print(f"Best validation cosine: {best_val:.4f}")
print(f"Best saved to: {SAVE_TRANSLATOR}")
print(f"Last saved to: {SAVE_LAST}")


[epoch 1] train_loss=0.000086  val_cos=0.9915  time=286.0s
  ↳ new best saved: ./checkpoints/diffusion_model_facilitator.pt  (best val_cos=0.9915)
[epoch 2] train_loss=0.000066  val_cos=0.9915  time=285.8s
  ↳ new best saved: ./checkpoints/diffusion_model_facilitator.pt  (best val_cos=0.9915)
[epoch 3] train_loss=0.000066  val_cos=0.9915  time=285.9s
[epoch 4] train_loss=0.000066  val_cos=0.9915  time=285.9s
  ↳ new best saved: ./checkpoints/diffusion_model_facilitator.pt  (best val_cos=0.9915)
[epoch 5] train_loss=0.000066  val_cos=0.9915  time=285.8s
[epoch 6] train_loss=0.000066  val_cos=0.9915  time=286.0s
[epoch 7] train_loss=0.000066  val_cos=0.9915  time=286.0s
  ↳ new best saved: ./checkpoints/diffusion_model_facilitator.pt  (best val_cos=0.9915)
[epoch 8] train_loss=0.000066  val_cos=0.9915  time=286.0s
[epoch 9] train_loss=0.000066  val_cos=0.9915  time=286.1s
[epoch 10] train_loss=0.000066  val_cos=0.9915  time=286.2s
Stage-2 training finished.
Best validation cosine: 0.9915

## Stage 3

Stage 3 fine-tunes a conditional diffusion model for peptide sequence generation. The input text is encoded with the SciBERT TextEncoder and then mapped by the Translator from Stage 2 to the conditioning representation used by the diffusion model. The target peptide is tokenized and encoded with the ProGen-based peptide encoder.

The training uses cross-entropy on token prediction together with an MSE term in the latent space. The diffusion backbone in the original repository is a Transformer-style denoiser created through `create_model_and_diffusion`, with embedding dimension 256 and sequence length 50.

A few implementation changes were needed to run this stage:

1. The conditioning block in `utils/train_util.py` was adjusted so the Translator output is used as `self_condition`.
2. Dataset paths in `utils/tokenizer.py` were adjusted for the Colab setup.

The released Stage 1 and Stage 2 checkpoints from the original authors were used for this stage. Because the full training schedule of the paper was not practical with the available resources, this notebook trains Stage 3 for 50 epochs as an implementation check rather than a full reproduction.


In [ ]:
from pathlib import Path
import re, shutil
REPO = Path("/content/TG-CDDPM")

In [ ]:
p = REPO / "utils" / "train_util.py"
src = p.read_text()
old_block_pat = (
    r"else:\n\s+with torch\.no_grad\(\):\n\s+"
    r"text_features = self\.text_encoder\(v\[i: i \+ self\.microbatch\]\.to\(self\.device\)\)\n\s+"
    r"text_features_norm = text_features / text_features\.norm\(dim=-1, keepdim=True\)\n\s+"
    r"text_features_norm = text_features_norm\.unsqueeze\(1\)\.repeat\(1, 50, 1\)\n\s+"
    r"#.*?\n\s+micro_cond\['self_condition'\] = text_features_norm"
)
new_block = (
    "else:\n"
    "                 with torch.no_grad():\n"
    "                         text_features = self.text_encoder(v[i: i + self.microbatch].to(self.device))\n"
    "                         text_features_norm = text_features / text_features.norm(dim=-1,keepdim=True)\n"
    "                         text_features_norm = text_features_norm.unsqueeze(1).repeat(1, 50, 1)\n"
    "                         timesteps = torch.tensor([0] * text_features.shape[0], device=self.device)\n"
    "                         fac_text_z = self.facilitator(inputs_embeds=text_features_norm,timesteps=timesteps)\n"
    "                         fac_text_z_norm = fac_text_z / fac_text_z.norm(dim=-1, keepdim=True)\n"
    "                         micro_cond['self_condition'] = text_features_norm\n"
)
src, n = re.subn(old_block_pat, new_block, src, flags=re.DOTALL)
if n == 0:
    raise RuntimeError("Did not find the expected text-conditioning block to replace in train_util.py")


In [ ]:
p = REPO / "utils" / "tokenizer.py"
src = p.read_text()
old_block_pat = ("'../dataset/backup/ensemble_train.jsonl'")
new_block = ("'/content/TG-CDDPM/dataset/backup/ensemble_train.jsonl'")
src, n = re.subn(old_block_pat, new_block, src, flags=re.DOTALL)
if n == 0:
    raise RuntimeError("Did not find the expected text-conditioning block to replace in train_util.py")
p.write_text(src)
print("Patched:", p)

p = REPO / "utils" / "tokenizer.py"
src = p.read_text()
old_block_pat = ("path = '../dataset/backup/ensemble_test.jsonl'")
new_block = ("path = '/content/TG-CDDPM/dataset/backup/ensemble_train.jsonl'")
src, n = re.subn(old_block_pat, new_block, src, flags=re.DOTALL)
if n == 0:
    raise RuntimeError("Did not find the expected text-conditioning block to replace in train_util.py")
p.write_text(src)
print("Patched:", p)

Patched: /content/TG-CDDPM/utils/tokenizer.py
Patched: /content/TG-CDDPM/utils/tokenizer.py


In [ ]:
p = REPO / "utils" / "train_util.py"
src = p.read_text()

# Grab only the forward_backward() body
fb_pat = r'(?ms)^[ \t]*def\s+forward_backward\s*\(self[^\)]*\)\s*:\s*\n(?P<body>.*?)(?=^[ \t]*def\s|\Z)'
m = re.search(fb_pat, src)
if not m:
    raise RuntimeError("forward_backward() not found in utils/train_util.py")

body = m.group("body")

# Your exact replacement
old_block_pat = r'^([ \t]*)print\s*\(\s*f?[\'"]Loss:.*?\)\s*$'
new_block = (
    r'\1self._ep_loss = getattr(self, "_ep_loss", 0.0) + float(loss.detach().item())'
    r'\n\1self._ep_count = getattr(self, "_ep_count", 0) + 1'
    r'\n\1if last_batch and ((self.step + 1) % self.log_interval == 0):'
    r'\n\1    avg = self._ep_loss / max(1, self._ep_count)'
    r'\n\1    print(f"[epoch {(self.step + 1)//self.log_interval}] avg_loss = {avg:.6f}")'
    r'\n\1    self._ep_loss = 0.0'
    r'\n\1    self._ep_count = 0'
)

body_new, n = re.subn(old_block_pat, new_block, body, flags=re.MULTILINE)
if n == 0:
    raise RuntimeError("Did not find the per-step Loss print to replace inside forward_backward().")

# Write the updated file (only the function body is changed)
src = src[:m.start("body")] + body_new + src[m.end("body"):]
p.write_text(src)
print(f"Patched exactly {n} occurrence in forward_backward(): {p}")


Patched exactly 1 occurrence in forward_backward(): /content/TG-CDDPM/utils/train_util.py


In [ ]:
import sys, os
if isinstance(REPO, Path):
    REPO = str(REPO)

os.chdir(REPO)
sys.path = [p if isinstance(p, str) else str(p) for p in sys.path]
if REPO not in sys.path:
    sys.path.insert(0, REPO)

import torch
import argparse
from transformers import AutoTokenizer
from train.TexPepAlignment import TextEncoder, PepEncoder
from config.backen_config import ProGenConfig
from model.backend import FacModel
from model.resample import create_named_schedule_sampler
from utils.tokenizer import load_data
from utils.script_utils import (
    model_and_diffusion_defaults, create_model_and_diffusion,
    args_to_dict, add_dict_to_argparser
)
from utils.train_util import TrainLoop


def create_argparser():
    defaults = dict(
        data_dir="./dataset",
        val_data_dir="./dataset",
        vocab_path='./mapping/vocab.txt',
        checkpoint_dir='./checkpoints',
        pretrained_diffusion='./checkpoints/w_pretraining_uncondition.pt',
        pretrained_text_encoder='./checkpoints/text_encoder.pt',
        pretrained_pep_encoder='./checkpoints/pep_encoder.pt',
        pretrained_translator='./checkpoints/translator.pt',
        timesteps=500,
        train_epoches=None,
        lr=1e-4,
        weight_decay=0.0,
        lr_anneal_steps=None,
        batch_size=16,
        microbatch=8,
        ema_rate="0.9999",
        seq_len=50,
        is_need_classifier=False,
        label_num=2,
        log_interval=None,
        save_interval=None,
        resume_checkpoint="",
        use_fp16=False,
        fp16_scale_growth=1e-3,
        schedule_sampler="uniform",
    )
    defaults.update(model_and_diffusion_defaults())
    p = argparse.ArgumentParser()
    add_dict_to_argparser(p, defaults)
    return p

args = create_argparser().parse_args(args=[])

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("creating model and diffusion...")

model, diffusion = create_model_and_diffusion(
    **args_to_dict(args, model_and_diffusion_defaults().keys())
)
model.to(device)
model.load_state_dict(torch.load(args.pretrained_diffusion, map_location=device))
schedule_sampler = create_named_schedule_sampler(args.schedule_sampler, diffusion)

pep_cfg = ProGenConfig(vocab_size=30,
                       n_positions=args.n_positions,
                       n_ctx=args.n_ctx,
                       n_embd=args.n_embd,
                       n_layer=2)

text_encoder = TextEncoder().to(device)
sd = torch.load(args.pretrained_text_encoder, map_location=device)
sd.pop("sci_bert.embeddings.position_ids", None)
text_encoder.load_state_dict(sd, strict=False)

pep_encoder = PepEncoder(pep_cfg).to(device)
pep_encoder.load_state_dict(torch.load(args.pretrained_pep_encoder, map_location=device), strict=False)

translator = FacModel(pep_cfg).to(device)
translator.load_state_dict(torch.load(args.pretrained_translator, map_location=device), strict=False)

# data
tok = AutoTokenizer.from_pretrained('./checkpoints/bert/prot_bert')
data = load_data(tok, args.seq_len, "train", args)

ProGenForCausalLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.


creating model and diffusion...


ProGenForCausalLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.


In [ ]:
EPOCHS = 50

num_examples = sum(1 for _ in open("./dataset/backup/ensemble_train.jsonl", 'r', encoding='utf-8'))
steps_per_epoch = num_examples // (args.batch_size*50)

args.log_interval = steps_per_epoch # print once per epoch
args.lr_anneal_steps = EPOCHS * steps_per_epoch
args.save_interval = steps_per_epoch  # or 5 * steps_per_epoch

print(f"steps_per_epoch = {steps_per_epoch} (Number of Batches)")

steps_per_epoch = 283 (Number of Batches)


In [ ]:
# train
print("training...")
TrainLoop(
    model=model,
    translator=translator,
    diffusion=diffusion,
    text_encoder=text_encoder,
    pep_encoder=pep_encoder,
    data=data,
    batch_size=args.batch_size,
    microbatch=args.microbatch,
    lr=args.lr,
    ema_rate=args.ema_rate,
    log_interval=args.log_interval,
    save_interval=args.save_interval,
    resume_checkpoint=args.resume_checkpoint,
    use_fp16=args.use_fp16,
    fp16_scale_growth=args.fp16_scale_growth,
    schedule_sampler=schedule_sampler,
    weight_decay=args.weight_decay,
    lr_anneal_steps=args.lr_anneal_steps,
).run_loop()

training...


Running tokenizer on public_database:   0%|          | 0/227064 [00:00<?, ? examples/s]

padding:   0%|          | 0/227064 [00:00<?, ? examples/s]

reload the random embeddings Embedding(30, 256)
reload the random embeddings Embedding(30, 256)
saving model 0...
saving model 0.9999...
[epoch 1] avg_loss = 0.029833
saving model 0...
saving model 0.9999...
[epoch 2] avg_loss = 0.023643
saving model 0...
saving model 0.9999...
[epoch 3] avg_loss = 0.022883
saving model 0...
saving model 0.9999...
[epoch 4] avg_loss = 0.021919
saving model 0...
saving model 0.9999...
[epoch 5] avg_loss = 0.020746
saving model 0...
saving model 0.9999...
[epoch 6] avg_loss = 0.019150
saving model 0...
saving model 0.9999...
[epoch 7] avg_loss = 0.017759
saving model 0...
saving model 0.9999...
[epoch 8] avg_loss = 0.017278
saving model 0...
saving model 0.9999...
[epoch 9] avg_loss = 0.016686
saving model 0...
saving model 0.9999...
[epoch 10] avg_loss = 0.016109
saving model 0...
saving model 0.9999...
[epoch 11] avg_loss = 0.015559
saving model 0...
saving model 0.9999...
[epoch 12] avg_loss = 0.015468
saving model 0...
saving model 0.9999...
[epoch 1

## Sampling

For the final sampling step, I used the checkpoints released by the original authors. My own models were not trained for enough epochs to make a fair comparison with the paper.

Using the prompt below, 10,000 peptide samples were generated. A subset of 1,000 samples is then used in the evaluation section.

**Sampling prompt:** `"This is a peptide: target inactive"`


In [ ]:
import sys, os
from pathlib import Path
REPO = Path("/content/TG-CDDPM")
if isinstance(REPO, Path):
    REPO = str(REPO)

os.chdir(REPO)
sys.path = [p if isinstance(p, str) else str(p) for p in sys.path]
if REPO not in sys.path:
    sys.path.insert(0, REPO)

import argparse
import jsonlines
import torch
import torch as th
from transformers import AutoTokenizer

from config.backen_config import ProGenConfig
from train.TexPepAlignment import TextEncoder, PepEncoder
from utils.tokenizer import TextTokenizer
from model.backend import FacModel
from utils.script_utils import (
    model_and_diffusion_defaults,
    create_model_and_diffusion,
    add_dict_to_argparser,
    args_to_dict,
)

# 1. Build parser with defaults
defaults = dict(
    clip_denoised=True,
    num_samples=1000,
    batch_size=200,
    max_loop=50,
    use_ddim=False,
    sample_path="./sample/",
    vocab_path="./mapping/vocab.txt",
    prior_path="./checkpoints/w_pretraining_w_inference.pt",
    model_path="./checkpoints/w_pretraining_w_inference.pt",
    pep_encoder_path="./checkpoints/pep_encoder.pt",
    text_encoder_path="./checkpoints/text_encoder.pt",
    translator_path="./checkpoints/translator.pt",
    classifier_scale=1.0,
    embedding_scale=0.0,
    use_fp16=False,
    classifier_use_fp16=False,
    seq_len=[50],
    vocab_size=30,
    class_cond=True,
    cls=2,
)
defaults.update(model_and_diffusion_defaults())

parser = argparse.ArgumentParser()
add_dict_to_argparser(parser, defaults)

# 2. Parse known args (ignoring Jupyter's -f flags)
args, unknown = parser.parse_known_args()

# 3. Device setup
device = "cuda" if th.cuda.is_available() else "cpu"
print("creating model and diffusion...")

# 4. Tokenizer definition
class PretainedTokenizer:
    def __init__(self):
        self.tokenizer = AutoTokenizer.from_pretrained("./checkpoints/bert/prot_bert")
        self.vocab = {}
        self.rev_vocab = {}
        with open("./checkpoints/bert/prot_bert/vocab.txt", "r") as f:
            for word in f:
                token = word.strip().split()[0]
                idx = len(self.vocab)
                self.vocab[token] = idx
                self.rev_vocab[idx] = token
        self.cls_index = self.vocab["[CLS]"]
        self.sep_index = self.vocab["[SEP]"]
        self.pad_index = self.vocab["[PAD]"]

    def batch_encode(self, seqs):
        return self.tokenizer(
            seqs,
            padding=True,
            truncation=True,
            return_tensors="pt",
            max_length=50,
        )["input_ids"]

    def batch_decode(self, indices):
        s = ""
        for j in indices:
            if j in (self.cls_index, self.pad_index):
                continue
            if j == self.sep_index:
                break
            s += self.rev_vocab[int(j)]
        return s

# 5. Load diffusion models
model, diffusion = create_model_and_diffusion(
    **args_to_dict(args, model_and_diffusion_defaults().keys())
)
model.load_state_dict(th.load(args.model_path, map_location="cpu"))
model.to(device)
if args.use_fp16:
    model.convert_to_fp16()
model.eval()

prior, _ = create_model_and_diffusion(
    **args_to_dict(args, model_and_diffusion_defaults().keys())
)
prior.load_state_dict(th.load(args.prior_path, map_location="cpu"))
prior.to(device)
if args.use_fp16:
    prior.convert_to_fp16()
prior.eval()

model_emb = th.nn.Embedding(
    num_embeddings=args.vocab_size,
    embedding_dim=args.n_embd,
    _weight=model.get_input_embeddings().weight.clone().cpu()
).eval().requires_grad_(False)

# 6. Prepare encoders with strict=False loading
config = ProGenConfig(
    vocab_size=args.vocab_size,
    n_positions=args.n_positions,
    n_ctx=args.n_ctx,
    n_embd=args.n_embd,
    n_layer=2,
)

# TextEncoder
text_tokenizer = TextTokenizer()
text_encoder = TextEncoder().to(device)
state = torch.load(args.text_encoder_path, map_location="cpu")
state.pop("sci_bert.embeddings.position_ids", None)
missing, unexpected = text_encoder.load_state_dict(state, strict=False)
if unexpected:
    print("Ignored keys in TextEncoder checkpoint:")
    for key in unexpected:
        print("  ", key)

# PepEncoder
pep_encoder = PepEncoder(config).to(device)
state = torch.load(args.pep_encoder_path, map_location="cpu")
missing, unexpected = pep_encoder.load_state_dict(state, strict=False)
if unexpected:
    print("Ignored keys in PepEncoder checkpoint:")
    for key in unexpected:
        print("  ", key)

# Translator (FacModel)
translator = FacModel(config).to(device)
state = torch.load(args.translator_path, map_location="cpu")
missing, unexpected = translator.load_state_dict(state, strict=False)
if unexpected:
    print("Ignored keys in Translator checkpoint:")
    for key in unexpected:
        print("  ", key)

# 7. Sampling helper fns
def cond_fn(inputs_embeds, timesteps, reference=None):
    assert reference is not None
    text_ids = reference[1]
    with th.enable_grad():
        x_in = inputs_embeds.detach().requires_grad_(True)
        pep_features = pep_encoder(None, x_in)
        text_features = text_encoder(text_ids)
        logits = text_features @ pep_features.t()
        return th.autograd.grad(logits.sum(), x_in)[0] * args.classifier_scale

def model_fn(inputs_embeds, timesteps, reference=None):
    assert reference is not None
    return model(
        inputs_embeds=inputs_embeds,
        timesteps=timesteps,
        self_condition=reference[0]
    )

# 8. Run sampling inline
print("sampling...")
myTokenizer = PretainedTokenizer()

for length in args.seq_len:
    all_peptides = []
    test_text = ["This is a peptide: target inactive"]
    test_tokens = text_tokenizer.batch_encode(test_text).repeat(args.batch_size, 1).to(device)

    while len(all_peptides) < args.num_samples:
        text_z = text_encoder(test_tokens)
        ref_text = text_z / text_z.norm(dim=-1, keepdim=True)
        timesteps = torch.zeros(args.batch_size, dtype=torch.long, device=device)
        fac_text_z = translator(inputs_embeds=ref_text, timesteps=timesteps)
        fac_text_z_norm = fac_text_z / fac_text_z.norm(dim=-1, keepdim=True)
        condition = fac_text_z_norm.unsqueeze(1).repeat(1, 50, 1)

        samples = (
            diffusion.p_sample_loop if not args.use_ddim else diffusion.ddim_sample_loop
        )(
            model_fn,
            noise=None,
            shape=(args.batch_size, length, args.n_embd),
            clip_denoised=args.clip_denoised,
            model_kwargs={"reference": [condition, test_tokens]},
            top_p=1,
            clamp_step=0,
            clamp_first=False,
            x_start=None,
            device=device,
            cond_fn=None,
        )

        logits = model.get_logits(samples[-1])
        _, indices = torch.topk(logits, k=1, dim=-1)
        indices = indices.squeeze(-1)
        all_peptides.extend([s.cpu().numpy() for s in indices])
        print(f"created {args.batch_size} samples")
        torch.cuda.empty_cache()

    os.makedirs(args.sample_path, exist_ok=True)
    with jsonlines.open(os.path.join(args.sample_path, "samples.jsonl"), "w") as writer:
        for seq in all_peptides:
            decoded = myTokenizer.batch_decode(seq)
            writer.write({"sequence": decoded})
            print(decoded)

    print(f"length {length} sampling complete")


ProGenForCausalLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.
ProGenForCausalLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't directly inherit from `GenerationMixin`. 

creating model and diffusion...


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/442M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/442M [00:00<?, ?B/s]

ProGenForCausalLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.


sampling...
created 200 samples
created 200 samples
created 200 samples
created 200 samples
created 200 samples
WLKRRIFRKLRARNLA
LRRLRRLLALRRHA
KLKVAVKKFASL
SIRLKLWLKKLHKKKLGSLL
ILLALLIRRFHKKKKIAKR
RRWLRLRRRRRKSI
SSRRSWVAVKILARWLHKKR
RLWGARRLLLLLALLPKKGARKKC
GVLLSALSALRRRWRRLRRRRL
IRLAKVKLLKGLFSKVLLKIGAS
KLLSLILALLKSKLKKLLNKL
KKFRIKLKLLAWLHTFFLNLVKKLI
FKRVLLIKKLRKIKL
KRRRVGARRRR
ILLSALKALKKRIGRLS
RRLEWLIKSLGKLRKLVAKLFAR
KLTFLRIVRRRLAHRRRRV
ARRGVRLVIVRSI
FHLLLKVLLLAALLKCARK
KRRLRKRRRLLVVVRRIKFLAKNAARLLLRRL
SLRRLLKLLLALLHRLAKKRVA
GLKSLLKLKFGLKRLKKN
FILKNLLKKLKIKSSFRLF
SLLFLLLKSLKVVVADL
ARNLLLAKIKKFLSAFVRLS
RKGLVNRRLAAKRILFRSFRPI
FWRLRKKRLKNLFKVF
FKRGRLKRRLGRAAAPAAIRKRA
RFLSILLRRLIKLKKRRLR
GFRFLAGHKKVHRGFVKKVALLALR
FFVRVFRLKVRRKKLGLRL
KKLIRLLRRK
GRIRRKLALRRRNRAEILFFLSLFRLRR
ILWGGPLSVPKRLRKLTGKGKKVRVCFGS
RLAFFLLRLLRRYL
WFLRLLLRVKKAVKLRN
FWKRIWALRLRAGKALRRLNIK
KRLVRRKRLK
LLKWFLKLLKLLALAKKWLKAL
KFKKLFKRKLGKLRRKIRS
RFLKLRLLKWRVRRLIWRLL
FFLARILRALVKLRSFVLSNNN
RIRLLRLRWSLFRRRGRASR
GLWLLLLGKLK
R

## Evaluation (AM-Score)

A subset of 1,000 generated peptides is evaluated with three AMP prediction tools: **amPEP**, **IPPF-FE**, and **CAMP**. For each tool, the AM-Score is the fraction of generated sequences predicted as antimicrobial peptides.


### Prepare generated samples for CAMP

The generated sequences are converted to FASTA format and submitted to the CAMP portal. The prediction output is then loaded again in the final evaluation step.


In [ ]:
import json, os, re
from pathlib import Path

IN_PATH = "/content/TG-CDDPM/sample/samples.jsonl"   #json sample path
OUT_FASTA = "./sample/test.fasta"
SEQ_KEYS = ["sequence","seq","peptide","trg","target","y"]
ID_KEYS  = ["id","name","uid","sid","pep_id","src"]
ALLOW = set("ACDEFGHIKLMNPQRSTVWYXBZJUO")  # common amino acids

def norm_seq(s: str):
    s = re.sub(r"\s+", "", s).upper()
    s = re.sub(r"[^A-Z]", "", s)
    return s

def extract_seq(obj):
    for k in SEQ_KEYS:
        if k in obj:
            return norm_seq(str(obj[k]))

    if all(isinstance(v, str) for v in obj.values()) and all(len(v) <= 100000 for v in obj.values()):

        k = next(iter(obj))
        return norm_seq(obj[k])
    return None

def extract_id(obj, idx):
    for k in ID_KEYS:
        if k in obj:
            return str(obj[k])
    return f"pep_{idx:06d}"

def iter_json_records(path):
    p = Path(path)
    if p.suffix.lower() == ".jsonl":
        with open(p, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                yield json.loads(line)
    else:
        with open(p, "r", encoding="utf-8") as f:
            data = json.load(f)
        if isinstance(data, dict) and all(isinstance(v, str) for v in data.values()):
            # dict of {id: seq}
            for i, (k,v) in enumerate(data.items()):
                yield {"id": k, "sequence": v}
        elif isinstance(data, list):
            for obj in data:
                yield obj
        else:

            yield data

def to_fasta(in_path, out_fasta):
    seen = set()
    kept = 0
    with open(out_fasta, "w", encoding="utf-8") as w:
        for i, obj in enumerate(iter_json_records(in_path), 1):
            seq = extract_seq(obj)
            if not seq:
                continue

            if not set(seq).issubset(ALLOW):

                cleaned = "".join(ch for ch in seq if ch in ALLOW)
                if len(cleaned) < 0.8*len(seq):
                    continue
                seq = cleaned
            pid = extract_id(obj, i)
            # de-dup on (seq)
            key = seq
            if key in seen:
                continue
            seen.add(key)
            kept += 1
            w.write(f">{pid}\n")
            # wrap lines to 60 chars (FASTA style)
            for j in range(0, len(seq), 60):
                w.write(seq[j:j+60] + "\n")
    return kept

n = to_fasta(IN_PATH, OUT_FASTA)
print(f"FASTA file is in {OUT_FASTA} with {n} unique sequences")


FASTA file is in ./sample/test.fasta with 1000 unique sequences


### Get amPEP predictions directly

The amPEP GitHub implementation is used locally instead of relying on a web portal. Its pretrained model is loaded, prediction scores are generated for `test.fasta`, and the scores are saved for the final evaluation.


In [ ]:
#  Install
%cd  /content
!pip -q install git+https://github.com/tlawrence3/amPEPpy.git biopython pandas

import os, re, json, pathlib, pandas as pd
from pathlib import Path

# INPUT (FASTA or JSON/JSONL)
IN_PATH = "/content/TG-CDDPM/sample/test.fasta"   # change this

def to_fasta(in_path, out_path="/content/axpep_input.fasta"):
    p = Path(in_path)
    def norm(s):
        s = re.sub(r"\s+","", s).upper()
        s = re.sub(r"[UZOBJ]","X", s)
        return re.sub(r"[^A-Z]","", s)
    if p.suffix.lower() in [".json",".jsonl"]:
        def iter_json():
            if p.suffix.lower()==".jsonl":
                for i,l in enumerate(open(p,encoding="utf-8"),1):
                    if l.strip():
                        o=json.loads(l)
                        yield f"pep_{i:06d}", next((o.get(k,"") for k in ["sequence","seq","peptide","trg","target","y"]), "")
            else:
                data=json.load(open(p,encoding="utf-8"))
                if isinstance(data,dict):
                    it=list(data.items())
                elif isinstance(data,list):
                    it=[(f"pep_{i:06d}", (x.get("sequence") or x.get("seq") or x.get("peptide") or x.get("trg") or x.get("target") or x.get("y",""))) for i,x in enumerate(data,1)]
                else:
                    it=[]
                for k,v in it: yield str(k), v
        with open(out_path,"w") as w:
            seen=set()
            for hid,seq in iter_json():
                s=norm(str(seq))
                if 4<=len(s)<=1000 and s not in seen:
                    seen.add(s); w.write(f">{hid}\n{s}\n")
        return out_path
    return in_path

FASTA = to_fasta(IN_PATH)

# ====== 2) Locate or fetch the amPEP model file ======
MODEL_PATH = None
try:
    import amPEPpy
    pkg = Path(amPEPpy.__file__).parent
    cands = list(pkg.rglob("*.model"))
    if cands:
        MODEL_PATH = str(cands[0])
except Exception:
    pass

if MODEL_PATH is None:
    # fallback: clone repo to get pretrained model

    !rm -rf /content/amPEPpy_repo
    !git clone -q https://github.com/tlawrence3/amPEPpy.git /content/amPEPpy_repo
    MODEL_PATH = "/content/amPEPpy_repo/pretrained_models/amPEP.model"
    assert os.path.exists(MODEL_PATH), "Could not find amPEP.model after cloning."

print("Using model:", MODEL_PATH)
print("Using FASTA:", FASTA)

# Run amPEP and compute AMscore
!ampep predict -m "$MODEL_PATH" -i "$FASTA" -o "/content/ampep_scores.tsv"

df = pd.read_csv("/content/ampep_scores.tsv", sep="\t")
prob_col = next(c for c in df.columns if any(k in c.lower() for k in ["prob","score","amp"]))
ams = (df[prob_col] >= 0.5).mean()
print(f"\nAMscore (amPEP, thr=0.5): {ams:.3f}  | N={len(df)}  | prob_col='{prob_col}'")
df.head(3)


/content
  Preparing metadata (setup.py) ... done
Using model: /content/amPEPpy_repo/pretrained_models/amPEP.model
Using FASTA: /content/TG-CDDPM/sample/test.fasta
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.4.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.4.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(

AMscore (amPEP, thr=0.5): 0.042  

,probability_nonAMP,probability_AMP,predicted,seq_id
0,0.255729,0.744271,AMP,pep_000001
1,0.323542,0.676458,AMP,pep_000002
2,0.416042,0.583958,AMP,pep_000003


In [ ]:
!ampep predict -i "$FASTA" -o "/content/ampep_scores.tsv"


Model file: amPEP.model not found!


### Get IPPF-FE predictions directly

The IPPF-FE repository is cloned and its pretrained pipeline is run on the 1,000 generated sequences. The model outputs prediction labels for the peptide samples, where label 1 represents an AMP prediction.


In [ ]:
%cd  /content
!git clone https://github.com/HanselYu/IPPF-FE
%cd  /content/IPPF-FE

Cloning into 'IPPF-FE'...
remote: Enumerating objects: 106, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 106 (delta 17), reused 26 (delta 11), pack-reused 70 (from 1)
Receiving objects: 100% (106/106), 4.61 MiB | 8.75 MiB/s, done.
Resolving deltas: 100% (30/30), done.


In [ ]:
!pip install -r requirment.txt
!pip -q install --upgrade pip
!pip -q install "torch==2.3.1" "torchvision==0.18.1" --index-url https://download.pytorch.org/whl/cu121

!pip -q install transformers sentencepiece pandas numpy scikit-learn lightgbm biopython
# 1) Make sure deps are fine (skip the repo's outdated requirment.txt)
pip -q install --upgrade transformers safetensors sentencepiece pandas numpy scikit-learn lightgbm


/content/IPPF-FE
  Using cached argon2_cffi-21.3.0-py3-none-any.whl.metadata (5.4 kB)
  Using cached argon2_cffi_bindings-21.2.0-cp36-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.7 kB)
  Using cached async_generator-1.10-py3-none-any.whl.metadata (4.9 kB)
  Using cached attrs-21.4.0-py2.py3-none-any.whl.metadata (9.8 kB)
  Using cached bleach-4.1.0-py2.py3-none-any.whl.metadata (24 kB)
  Using cached certifi-2021.5.30-py2.py3-none-any.whl.metadata (3.0 kB)
  Using cached cffi-1.15.0.tar.gz (484 kB)
  Preparing metadata (setup.py) ... done
  Using cached charset_normalizer-2.0.9-py3-none-any.whl.metadata (11 kB)
  Using cached click-8.0.3-py3-none-any.whl.metadata (3.2 kB)
  Using cached cycler-0.11.0-py3-none-any.whl.metadata (785 bytes)
ERROR: Ignored the following versions that require a different python version: 0.7 Requires-Python >=3.6,<3.7; 0.8 Requires-Python >=3.6,<3.7
ERROR: Could not find a version that satisfies the requirement dataclasses==0.8 (from versi

In [ ]:
%cd /content/IPPF-FE
!python Pantibacterial.py Train -

Streaming output truncated to the last 5000 lines.
For sequence  7945
For sequence  7946
For sequence  7947
For sequence  7948
For sequence  7949
For sequence  7950
For sequence  7951
For sequence  7952
For sequence  7953
For sequence  7954
For sequence  7955
For sequence  7956
For sequence  7957
For sequence  7958
For sequence  7959
For sequence  7960
For sequence  7961
For sequence  7962
For sequence  7963
For sequence  7964
For sequence  7965
For sequence  7966
For sequence  7967
For sequence  7968
For sequence  7969
For sequence  7970
For sequence  7971
For sequence  7972
For sequence  7973
For sequence  7974
For sequence  7975
For sequence  7976
For sequence  7977
For sequence  7978
For sequence  7979
For sequence  7980
For sequence  7981
For sequence  7982
For sequence  7983
For sequence  7984
For sequence  7985
For sequence  7986
For sequence  7987
For sequence  7988
For sequence  7989
For sequence  7990
For sequence  7991
For sequence  7992
For sequence  7993
For sequence  7994

In [ ]:
from pathlib import Path
import re

p = Path("/content/IPPF-FE/Pantibacterial.py")
s = p.read_text()

s = s.replace('"prot_t5_xl_uniref50"', '"Rostlab/prot_t5_xl_uniref50"')

s = s.replace(
    'T5EncoderModel.from_pretrained("Rostlab/prot_t5_xl_uniref50")',
    'T5EncoderModel.from_pretrained("Rostlab/prot_t5_xl_uniref50", use_safetensors=True, torch_dtype=torch.float16)'
)

s = s.replace(
    "torch.device('cuda:3' if torch.cuda.is_available() else 'cpu')",
    "torch.device('cuda' if torch.cuda.is_available() else 'cpu')"
)

if "import numpy as np" not in s:
    s = s.replace("import sys", "import sys\nimport numpy as np")
if "import pandas as pd" not in s:
    s = s.replace("import sys", "import sys\nimport pandas as pd")

p.write_text(s)
print("Patched Pantibacterial.py")
print("Leftover bad IDs:", re.findall(r"prot_t5_xl_uniref50", s))


In [ ]:
!python Pantibacterial.py Predict /content/TG-CDDPM/sample/test.fasta

2025-09-20 06:30:50.475865: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1758349850.497014   27381 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1758349850.503609   27381 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1758349850.520005   27381 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1758349850.520033   27381 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1758349850.520036   27381 computation_placer.cc:177] computation placer alr

## Final results

The final step combines predictions from CAMP, IPPF-FE, and the amPEP-based evaluation. The AM-Score for each method is calculated from its binary predictions on 1,000 generated samples. A combined average is also reported.

These numbers are used as a reproduction check for the pipeline. Since the full TG-CDDPM training schedule was not reproduced and author checkpoints were used in later stages, they should not be interpreted as a direct reproduction of the paper's final benchmark.


In [ ]:
import pandas  as pd
ippf_fe_output = pd.read_excel("/content/IPPF-FE/Pre_label.xlsx")
Camp_output = pd.read_csv("/content/CAMPdownload_2025-09-20 12-02-33.txt",sep='\t')
Axpep = pd.read_csv("/content/ampep_scores.tsv",sep='\t')

In [ ]:
final_results = {'CAMP': [(pd.get_dummies(Camp_output['Class'], dtype=int)['AMP']).mean()],
 'IPPF-FE': [(ippf_fe_output['Pre_label']).mean()],
 'AxPEP': [(pd.get_dummies(Axpep['predicted'], dtype=int)['AMP']).mean()]}
final_results = pd.DataFrame(final_results,index=['AM-Score for 1000 samples'])
final_results['Total'] = final_results.mean(axis=1)

In [ ]:
display(final_results.T)

,AM-Score for 1000 samples
CAMP,1.000
IPPF-FE,0.985
AxPEP,0.958
Total,0.981
